In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# Load train and test sets
train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

features = [c for c in train.columns if c != "target"]
X_train, y_train = train[features], train["target"]
X_test,  y_test  = test[features],  test["target"]

print(f"Train: {X_train.shape[0]:,} rows | {len(features)} features")
print(f"Test:  {X_test.shape[0]:,} rows")
print(f"Features: {features}")
print(f"Train default rate: {y_train.mean():.2%}")

Train: 1,109,330 rows | 5 features
Test:  277,333 rows
Features: ['int_rate', 'dti', 'loan_amnt', 'annual_inc', 'revol_util']
Train default rate: 19.82%


In [2]:
# Pipeline: StandardScaler + Logistic Regression
# class_weight=balanced handles the 80/20 good/bad imbalance automatically
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

pipeline.fit(X_train, y_train)
print("Model trained successfully")

# Predict probabilities on test set
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Quick AUC check
auc = roc_auc_score(y_test, y_prob)
print(f"\nAUC on test set: {auc:.4f}")
print(f"Gini coefficient: {(2 * auc - 1):.4f}")

Model trained successfully

AUC on test set: 0.6901
Gini coefficient: 0.3802


In [3]:
def prob_to_score(prob, min_score=300, max_score=850):
    """
    Convert default probability to credit score.
    Higher score = lower risk — same direction as FICO.
    """
    score = min_score + (max_score - min_score) * (1 - prob)
    return np.round(score).astype(int)

# Apply scoring to test set
results = pd.DataFrame({
    "actual":          y_test.values,
    "pd_probability":  y_prob,
    "credit_score":    prob_to_score(y_prob)
})

print(results.head(10))
print(f"\nGood loans — avg score: {results[results['actual']==0]['credit_score'].mean():.0f}")
print(f"Bad loans  — avg score: {results[results['actual']==1]['credit_score'].mean():.0f}")
print(f"\nScore separation: {results[results['actual']==0]['credit_score'].mean() - results[results['actual']==1]['credit_score'].mean():.0f} points")

   actual  pd_probability  credit_score
0       0        0.689970           471
1       0        0.612829           513
2       0        0.515583           566
3       0        0.274493           699
4       0        0.502733           573
5       0        0.616143           511
6       0        0.287129           692
7       0        0.454699           600
8       0        0.678128           477
9       0        0.511439           569

Good loans — avg score: 604
Bad loans  — avg score: 546

Score separation: 59 points


In [4]:
# Save trained model  actual  pd_probability  credit_score
0       0        0.689970           471
1       0        0.612829           513
2       0        0.515583           566
3       0        0.274493           699
4       0        0.502733           573
5       0        0.616143           511
6       0        0.287129           692
7       0        0.454699           600
8       0        0.678128           477
9       0        0.511439           569

Good loans — avg score: 604
Bad loans  — avg score: 546

with open("../models/scorecard_v1.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# Save results for validation notebook
results.to_csv("../data/processed/test_results.csv", index=False)

print("Model saved → models/scorecard_v1.pkl")
print("Results saved → data/processed/test_results.csv")
print(f"\nFinal AUC: {auc:.4f}")
print(f"Final Gini: {(2*auc-1):.4f}")
print("\nModel complete — move to 04_validation.ipynb")

Model saved → models/scorecard_v1.pkl
Results saved → data/processed/test_results.csv

Final AUC: 0.6901
Final Gini: 0.3802

Model complete — move to 04_validation.ipynb


In [11]:
# === BOOST MODEL WITH ENGINEERED FEATURES ===

# Reload original train and test
train_orig = pd.read_csv("../data/processed/loans_clean.csv")

# Re-run the same cleaning from NB2
train_orig["int_rate"] = (train_orig["int_rate"]
                          .astype(str)
                          .str.replace("%","",regex=False)
                          .astype(float))
train_orig["term"] = (train_orig["term"]
                      .astype(str)
                      .str.extract(r"(\d+)")[0]
                      .astype(float))
train_orig["emp_length"] = (train_orig["emp_length"]
                            .astype(str)
                            .str.extract(r"(\d+)")[0]
                            .astype(float))
train_orig["revol_util"] = (train_orig["revol_util"]
                            .astype(str)
                            .str.replace("%","",regex=False)
                            .astype(float))

# Fill missing
for col in ["annual_inc","dti","revol_util","emp_length"]:
    train_orig[col] = train_orig[col].fillna(train_orig[col].median())

# Cap outliers
train_orig["revol_util"] = train_orig["revol_util"].clip(upper=100)
train_orig["annual_inc"] = train_orig["annual_inc"].clip(
    upper=train_orig["annual_inc"].quantile(0.99))
train_orig["dti"] = train_orig["dti"].clip(upper=60)

# === NEW ENGINEERED FEATURES ===
# Monthly debt burden relative to income
train_orig["debt_to_inc_monthly"] = (
    (train_orig["dti"] / 100 * train_orig["annual_inc"]) / 12
)

# Loan amount relative to annual income
train_orig["loan_to_income"] = (
    train_orig["loan_amnt"] / (train_orig["annual_inc"] + 1)
)

# Interest rate times DTI — combined stress measure
train_orig["rate_x_dti"] = train_orig["int_rate"] * train_orig["dti"]

# High utilisation flag — above 80% is a stress signal
train_orig["high_util_flag"] = (
    (train_orig["revol_util"] > 80).astype(int)
)

# Term is 60 months flag — longer loans default more
train_orig["is_60_month"] = (
    (train_orig["term"] == 60).astype(int)
)

print("Engineered features created")

# New feature set
new_features = [
    "int_rate", "dti", "loan_amnt", "annual_inc", "revol_util",
    "emp_length", "open_acc", "total_acc", "delinq_2yrs",
    "debt_to_inc_monthly", "loan_to_income",
    "rate_x_dti", "high_util_flag", "is_60_month"
]

# Check all features exist
available = [f for f in new_features if f in train_orig.columns]
print(f"Available features: {len(available)}")
print(available)

Engineered features created
Available features: 14
['int_rate', 'dti', 'loan_amnt', 'annual_inc', 'revol_util', 'emp_length', 'open_acc', 'total_acc', 'delinq_2yrs', 'debt_to_inc_monthly', 'loan_to_income', 'rate_x_dti', 'high_util_flag', 'is_60_month']


In [12]:
# Check which features have NaN
nan_check = pd.DataFrame({
    "NaN Count": X2.isnull().sum(),
    "NaN %": (X2.isnull().sum() / len(X2) * 100).round(2)
})
print("Features with NaN values:")
print(nan_check[nan_check["NaN Count"] > 0])

# Fix 1 — fill NaN in engineered features with median
for col in available:
    if X2[col].isnull().sum() > 0:
        median_val = X2[col].median()
        X2[col] = X2[col].fillna(median_val)
        print(f"  Fixed {col}: filled with median {median_val:.4f}")

# Fix 2 — replace any infinity values created by division
X2 = X2.replace([np.inf, -np.inf], np.nan)
for col in available:
    if X2[col].isnull().sum() > 0:
        X2[col] = X2[col].fillna(X2[col].median())

# Verify clean
print(f"\nNaN remaining: {X2.isnull().sum().sum()}")
print(f"Inf remaining: {np.isinf(X2.values).sum()}")
print("Data is clean — ready to train")

Features with NaN values:
             NaN Count  NaN %
open_acc             3    0.0
total_acc            3    0.0
delinq_2yrs          3    0.0
  Fixed open_acc: filled with median 11.0000
  Fixed total_acc: filled with median 23.0000
  Fixed delinq_2yrs: filled with median 0.0000

NaN remaining: 0
Inf remaining: 0
Data is clean — ready to train


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

# Clean X2 one final time before splitting
X2 = X2.replace([np.inf, -np.inf], np.nan)
X2 = X2.fillna(X2.median())

# Verify clean
print(f"NaN in X2 before split: {X2.isnull().sum().sum()}")

# Split
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print(f"NaN in X2_train: {X2_train.isnull().sum().sum()}")
print(f"NaN in X2_test:  {X2_test.isnull().sum().sum()}")

# Pipeline now includes SimpleImputer as safety net
# This catches any NaN that slips through — belt and braces approach
pipeline_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        solver="lbfgs",
        C=0.1
    ))
])

pipeline_v2.fit(X2_train, y2_train)

# Evaluate
y_prob_v2 = pipeline_v2.predict_proba(X2_test)[:, 1]
auc_v2    = roc_auc_score(y2_test, y_prob_v2)
gini_v2   = 2 * auc_v2 - 1

print("=" * 40)
print("   MODEL V2 RESULTS")
print("=" * 40)
print(f"  AUC:  {auc_v2:.4f}  (was 0.6901)")
print(f"  Gini: {gini_v2:.4f}  (was 0.3802)")
print(f"  Improvement: +{(auc_v2 - 0.6901):.4f} AUC")
print("=" * 40)

# Score results
results_v2 = pd.DataFrame({
    "actual":         y2_test.values,
    "pd_probability": y_prob_v2,
    "credit_score":   prob_to_score(y_prob_v2)
})

good_avg = results_v2[results_v2["actual"]==0]["credit_score"].mean()
bad_avg  = results_v2[results_v2["actual"]==1]["credit_score"].mean()

print(f"\nGood loans avg score: {good_avg:.0f}")
print(f"Bad loans  avg score: {bad_avg:.0f}")
print(f"Score separation:     {good_avg - bad_avg:.0f} points")

NaN in X2 before split: 0
NaN in X2_train: 0
NaN in X2_test:  0
   MODEL V2 RESULTS
  AUC:  0.6968  (was 0.6901)
  Gini: 0.3936  (was 0.3802)
  Improvement: +0.0067 AUC

Good loans avg score: 606
Bad loans  avg score: 543
Score separation:     63 points


In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# XGBoost handles NaN natively — no imputer needed
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y2_train==0).sum() / (y2_train==1).sum(),
    random_state=42,
    eval_metric="auc",
    verbosity=0
)

print("Training XGBoost — please wait ~2 minutes...")
xgb_model.fit(
    X2_train, y2_train,
    eval_set=[(X2_test, y2_test)],
    verbose=False
)

# Evaluate
y_prob_xgb = xgb_model.predict_proba(X2_test)[:, 1]
auc_xgb    = roc_auc_score(y2_test, y_prob_xgb)
gini_xgb   = 2 * auc_xgb - 1

print("=" * 45)
print("   XGBOOST MODEL RESULTS")
print("=" * 45)
print(f"  AUC:  {auc_xgb:.4f}  (Logistic: 0.6968)")
print(f"  Gini: {gini_xgb:.4f}  (Logistic: 0.3936)")
print(f"  Improvement over LR: +{(auc_xgb-0.6968):.4f}")
print("=" * 45)

# Score results
results_xgb = pd.DataFrame({
    "actual":         y2_test.values,
    "pd_probability": y_prob_xgb,
    "credit_score":   prob_to_score(y_prob_xgb)
})

good_avg = results_xgb[results_xgb["actual"]==0]["credit_score"].mean()
bad_avg  = results_xgb[results_xgb["actual"]==1]["credit_score"].mean()

print(f"\nGood loans avg score: {good_avg:.0f}")
print(f"Bad loans  avg score: {bad_avg:.0f}")
print(f"Score separation:     {good_avg - bad_avg:.0f} points")

Training XGBoost — please wait ~2 minutes...
   XGBOOST MODEL RESULTS
  AUC:  0.7092  (Logistic: 0.6968)
  Gini: 0.4184  (Logistic: 0.3936)
  Improvement over LR: +0.0124

Good loans avg score: 611
Bad loans  avg score: 539
Score separation:     72 points


In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
import numpy as np

# Tuned XGBoost — more trees, deeper, more regularisation
xgb_tuned = XGBClassifier(
    # More estimators — diminishing returns after 500
    n_estimators=500,
    # Deeper trees capture more complex interactions
    max_depth=6,
    # Slower learning rate with more trees = better generalisation
    learning_rate=0.03,
    # Subsampling prevents overfitting
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    # L1 and L2 regularisation
    reg_alpha=0.1,
    reg_lambda=1.5,
    # Handle class imbalance
    scale_pos_weight=(y2_train==0).sum() / (y2_train==1).sum(),
    # Minimum samples per leaf — prevents overfitting
    min_child_weight=50,
    random_state=42,
    eval_metric="auc",
    verbosity=0
)

print("Training tuned XGBoost — please wait 3-5 minutes...")
xgb_tuned.fit(
    X2_train, y2_train,
    eval_set=[(X2_test, y2_test)],
    verbose=100
)

y_prob_tuned = xgb_tuned.predict_proba(X2_test)[:, 1]
auc_tuned    = roc_auc_score(y2_test, y_prob_tuned)
gini_tuned   = 2 * auc_tuned - 1

print("=" * 45)
print("   TUNED XGBOOST RESULTS")
print("=" * 45)
print(f"  AUC:  {auc_tuned:.4f}  (previous: 0.7092)")
print(f"  Gini: {gini_tuned:.4f}  (previous: 0.4184)")
print(f"  Change: {(auc_tuned - 0.7092):+.4f}")
print("=" * 45)

Training tuned XGBoost — please wait 3-5 minutes...
[0]	validation_0-auc:0.69301
[100]	validation_0-auc:0.70435
[200]	validation_0-auc:0.70741
[300]	validation_0-auc:0.70869
[400]	validation_0-auc:0.70946
[499]	validation_0-auc:0.71005
   TUNED XGBOOST RESULTS
  AUC:  0.7100  (previous: 0.7092)
  Gini: 0.4201  (previous: 0.4184)
  Change: +0.0008


In [17]:
# === ADDITIONAL CREDIT RISK FEATURES ===

# Instalment to income ratio — monthly payment burden
train_orig["installment_to_inc"] = (
    train_orig["loan_amnt"] / 
    (train_orig["annual_inc"] / 12 + 1)
)

# Revolving balance to income
train_orig["revol_to_income"] = (
    train_orig["revol_bal"] / 
    (train_orig["annual_inc"] + 1)
).clip(upper=10)

# Total accounts to open accounts ratio — credit utilisation breadth
train_orig["open_to_total_acc"] = (
    train_orig["open_acc"] / 
    (train_orig["total_acc"] + 1)
)

# Delinquency flag — any delinquency in last 2 years
train_orig["has_delinq"] = (
    (train_orig["delinq_2yrs"] > 0).astype(int)
)

# Risk tier from int_rate — mimics credit grade
train_orig["rate_tier"] = pd.cut(
    train_orig["int_rate"],
    bins=[0, 8, 12, 16, 20, 100],
    labels=[1, 2, 3, 4, 5]
).astype(float)

# Extended feature set
extended_features = [
    # Original IV-selected
    "int_rate", "dti", "loan_amnt", "annual_inc", "revol_util",
    # From V2
    "emp_length", "open_acc", "total_acc", "delinq_2yrs",
    "debt_to_inc_monthly", "loan_to_income",
    "rate_x_dti", "high_util_flag", "is_60_month",
    # New additions
    "installment_to_inc", "revol_to_income",
    "open_to_total_acc", "has_delinq", "rate_tier",
    # Raw balance
    "revol_bal"
]

available_ext = [f for f in extended_features 
                 if f in train_orig.columns]
print(f"Extended feature set: {len(available_ext)} features")
print(available_ext)

# Build clean X and y
X3 = train_orig[available_ext].copy()
y3 = train_orig["target"]

# Clean
X3 = X3.replace([np.inf, -np.inf], np.nan)
X3 = X3.fillna(X3.median())

print(f"\nNaN in X3: {X3.isnull().sum().sum()}")
print(f"Shape: {X3.shape}")

Extended feature set: 20 features
['int_rate', 'dti', 'loan_amnt', 'annual_inc', 'revol_util', 'emp_length', 'open_acc', 'total_acc', 'delinq_2yrs', 'debt_to_inc_monthly', 'loan_to_income', 'rate_x_dti', 'high_util_flag', 'is_60_month', 'installment_to_inc', 'revol_to_income', 'open_to_total_acc', 'has_delinq', 'rate_tier', 'revol_bal']

NaN in X3: 0
Shape: (1386663, 20)


In [18]:
from sklearn.model_selection import train_test_split

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)

xgb_v3 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    reg_alpha=0.1,
    reg_lambda=1.5,
    scale_pos_weight=(y3_train==0).sum() / (y3_train==1).sum(),
    min_child_weight=50,
    random_state=42,
    eval_metric="auc",
    verbosity=0
)

print("Training V3 on extended features — please wait...")
xgb_v3.fit(
    X3_train, y3_train,
    eval_set=[(X3_test, y3_test)],
    verbose=100
)

y_prob_v3 = xgb_v3.predict_proba(X3_test)[:, 1]
auc_v3    = roc_auc_score(y3_test, y_prob_v3)
gini_v3   = 2 * auc_v3 - 1

good_avg = results_xgb[results_xgb["actual"]==0]["credit_score"].mean()
bad_avg  = results_xgb[results_xgb["actual"]==1]["credit_score"].mean()

print("=" * 45)
print("   MODEL V3 — EXTENDED FEATURES")
print("=" * 45)
print(f"  Features:  {len(available_ext)}")
print(f"  AUC:       {auc_v3:.4f}  (V2: 0.7092)")
print(f"  Gini:      {gini_v3:.4f}  (V2: 0.4184)")
print(f"  Change:    {(auc_v3 - 0.7092):+.4f} AUC")
print("=" * 45)

# Save best model
import pickle
with open("../models/scorecard_final.pkl", "wb") as f:
    pickle.dump(xgb_v3, f)

results_v3 = pd.DataFrame({
    "actual":         y3_test.values,
    "pd_probability": y_prob_v3,
    "credit_score":   prob_to_score(y_prob_v3)
})
results_v3.to_csv("../data/processed/test_results.csv", index=False)

good_avg = results_v3[results_v3["actual"]==0]["credit_score"].mean()
bad_avg  = results_v3[results_v3["actual"]==1]["credit_score"].mean()
print(f"\nScore separation: {good_avg - bad_avg:.0f} points")
print("Final model saved — move to 04_validation.ipynb")

Training V3 on extended features — please wait...
[0]	validation_0-auc:0.69457
[100]	validation_0-auc:0.70640
[200]	validation_0-auc:0.70929
[300]	validation_0-auc:0.71059
[400]	validation_0-auc:0.71138
[499]	validation_0-auc:0.71196
   MODEL V3 — EXTENDED FEATURES
  Features:  20
  AUC:       0.7120  (V2: 0.7092)
  Gini:      0.4239  (V2: 0.4184)
  Change:    +0.0028 AUC

Score separation: 74 points
Final model saved — move to 04_validation.ipynb
